# DataQualityProfiler

In [16]:
import os

from crewai import Agent, Crew, Process, Task, LLM
from crewai.project import CrewBase, agent, crew, task
from crewai_tools import FileReadTool, CodeInterpreterTool
from datetime import datetime
from pathlib import Path

import warnings
warnings.filterwarnings("ignore", category=SyntaxWarning, module="pysbd")

from dotenv import load_dotenv
load_dotenv()

True

In [17]:
DOCKER_IMAGE = str(os.getenv("DOCKER_IMAGE"))
SAMPLE_FILE = str(os.getenv("SAMPLE_DATA"))
DATE = datetime.now().strftime('%Y%m%d_%H%M%S')
DATA_DIR = Path(str(os.getenv('DATA_DIR')))
OUTPUT_DIR = Path(str(os.getenv('OUTPUT_DIR')))


## LLM

In [18]:
llm_writer = LLM(model=f"{str(os.getenv('MODEL'))}",base_url=str(os.getenv("API_BASE"))) 
llm_coder = LLM(model=str(os.getenv('CODER_MODEL')), base_url=str(os.getenv("API_BASE")))

LiteLLM is not available, falling back to LiteLLM


ImportError: Fallback to LiteLLM is not available

## AGENTS

In [ ]:
loader_config={
  "role": "Dataset Loader & Structural Inspector",
  "goal": """
    Validate the dataset_path BEFORE loading. 
    If invalid, return a valid DatasetLoadOut JSON containing warnings and no data.
    Never guess or create synthetic data if loading fails.
    If valid, load the dataset reliably, infer schema, types, structural metadata, and sample rows
    WITHOUT computing any statistics or interpretations.
    """,
  "backstory": """
    You specialize in dataset structural analysis. 
    You never guess, never compute statistics,
    and never infer patterns beyond schema-level metadata.
    """
}

profiler_config={
  "role": "Data Quality Profiler & Code-Execution Analyst",
  "goal": """
    Compute missing values, duplicates, and summary statistics STRICTLY through Python
    code execution. All metrics must be correct, numeric, and reproducible.
    """,
  "backstory": """
    You are a precise analyst who relies entirely on verified numeric computation.
    You reject ambiguous or unstructured results and you do not produce narrative insights.
    """
}

reporter_config={
  "role": "Markdown Data Quality Reporter",
  "goal": """
    Transform profiler outputs into a deterministic Markdown report following a fixed schema:
      1. Overview
      2. Schema Summary
      3. Missingness
      4. Duplicates
      5. Summary Statistics
      6. Data Quality Score
      7. Recommendations
    You do not compute metrics.
    """,
  "backstory": """
    You write clean, highly structured reports with no hallucinated metrics or columns.
    You only describe what exists in the profiler JSON.
    """
}

In [ ]:
loader_agent = Agent(
    role=loader_config.get("role"),
    goal=loader_config.get("goal"),
    backstory=loader_config.get("goal"),
    llm=llm_writer,
    tools=[FileReadTool()],
    short_term_memory=True,
    verbose=True
)

profiler_agent = Agent(
    role=profiler_config.get("role"),
    goal=profiler_config.get("goal"),
    backstory=profiler_config.get("goal"),
    llm=llm_coder,
    tools=[CodeInterpreterTool(
        image=DOCKER_IMAGE,
        timeout=300,
        # memory_limit="2g",
    )],
    allow_code_execution=True,
    code_execution_mode="safe",
    max_execution_time=300,
    max_retry_limit=3,
    short_term_memory=True,
    verbose=True
)

reporter_agent = Agent(
    role=reporter_config.get("role"),
    goal=reporter_config.get("goal"),
    backstory=reporter_config.get("goal"),
    llm=llm_writer,
    short_term_memory=True,
    verbose=True
)

## MODELS

In [5]:
from pydantic import BaseModel, Field
from typing import List, Dict, Any
from datetime import datetime


class DatasetLoadOut(BaseModel):
    dataset_name: str
    generated_at: datetime

    row_count: int
    column_count: int
    
    columns: List[str]
    inferred_types: Dict[str, str]
    sample_rows: List[Dict[str, Any]]
    memory_usage_bytes: int

    warnings: List[str] = Field(default=[])
    validation_errors: List[str] = []
    validation_info: Dict[str, Any] = Field(default={}, description="Metadata about validation state")


class DatasetProfile(BaseModel):
    dataset_name: str = Field(..., description="Dataset name captured by load_dataset")
    generated_at: datetime

    missing: Dict[str, float]
    duplicates: int

    # enforce structured stats (prevents hallucinated keys)
    summary_stats: Dict[str, Dict[str, float]]
    data_quality_score: float
    score_components: Dict[str, float] = Field(default={}, description="ex: missingness_penalty, duplicate_penalty,...")

    class_distribution: Dict[str, int] = {}
    imbalance_ratio: float = 0.0
    bias_flags: List[str] = []

    warnings: List[str] = []
    validation_errors: List[str] = []

    processing_time_seconds: float = 0.0
    warning_count: int = 0
    num_columns_profiled: int = 0
    retry_count: int = 0

## TOOLS

No tools.

## TASKS

In [ ]:
load_dataset_config={
  "description": '''
    Load dataset from {dataset_path} (do not copy/move the file).

    Validation (stop on first fatal error):
      - file_exists: path exists
      - is_csv: file extension is ".csv"
      - readable: accessible (no permission error)
      - parseable: pandas can read (pd.read_csv) without exception
      - non_empty: row_count > 0
      - column_limit: column_count <= 2000
      - large_dataset: if row_count > 5_000_000, set truncated=true and add warning "large_dataset_truncated"

    Failure behavior (any fatal validation fails):
      - Do NOT load/process the dataset further
      - Return DatasetLoadOut JSON with:
          - empty structures (columns=[], inferred_types={}, sample_rows=[])
          - row_count=0, column_count=0, memory_usage_bytes=0
          - warnings populated
          - validation_errors populated (fatal reasons)
          - validation_info populated (see below)

    Success behavior:
      - Extract:
          - dataset_name from {dataset_path}
          - row_count, column_count
          - columns (ordered)
          - inferred_types (pandas dtype strings)
          - sample_rows (max 5 records, JSON-serializable)
          - memory_usage_bytes (df.memory_usage(deep=True).sum())

    Required tracking fields:
      - warnings: list[str] (non-fatal issues + policy violations)
      - validation_errors: list[str] (fatal validation failures)
      - validation_info (structured booleans):
          file_exists: true/false
          readable: true/false
          parseable: true/false
          truncated: true/false
          too_many_columns: true/false

    Guardrail:
      - Loader must NOT compute or emit any statistics (missingness, duplicates, describe, averages, etc).
      - If any stats-like fields appear in output, remove them and add warning "loader_attempted_statistics".
    ''',
  "expected_output": '''
    JSON matching DatasetLoadOut schema.''',
  "agent": "loader",
  "output_file": "output/{outdata_dir}/loaded.json"
  # output_pydantic: dataset_profiler.models.load_models.DatasetLoadOut
  }

profile_dataset_config={
  "description": '''
    Profile dataset quality using the output of profile_dataset.

    Skip conditions:
      - If load_dataset contains fatal validation_errors OR warnings indicating critical failure
        (e.g., file not found, unreadable/unparseable, zero rows, too many columns),
        do NOT run profiling. Return DatasetProfile JSON with:
          - missing={}, duplicates=0, summary_stats={}
          - data_quality_score=0
          - warnings propagated from load_dataset (+ "profiling_skipped_due_to_validation" optional)

    Otherwise:
      - Compute (via CodeInterpreterTool only using Docker image; raw Python code string; JSON-serializable outputs):
          - missing: per-column missing fraction (df.isna().mean().to_dict())
          - duplicates: int(df.duplicated().sum())
          - summary_stats: df.describe(percentiles=[0.25, 0.5, 0.75]).to_dict()
      - Data Quality Score:
          missingness_penalty = avg_missing * 100
          duplicate_penalty   = min((duplicates / row_count) * 100, 100)
          score = 100 - (missingness_penalty * 0.6 + duplicate_penalty * 0.4)
          clamp to [0, 100]

    Optional target checks (only if a target column is explicitly present/known):
      - value_counts for target
      - imbalance_ratio = largest_class / total_rows
      - if imbalance_ratio > 0.80 add warning "severe_class_imbalance"

    Timing:
      - record start_time and end_time (Python)
      - compute processing_time_seconds inside Docker execution

    Implementation constraint:
      - Use a single consolidated Python script (one tool call) that reads the CSV and computes all metrics.
      - Return only JSON (no narrative). If any interpretation appears, remove it and add warning
        "profiler_attempted_interpretation".
    ''',
  "expected_output": '''
    JSON matching DatasetProfile schema.
    ''',
  # "agent": "profiler",
  # "context": [load_dataset],
  "output_file": "output/{outdata_dir}/profile.json"
  # output_pydantic: dataset_profiler.models.profile_models.DatasetProfile
}

write_report_config={
  "description": '''
    Write a deterministic Markdown report using ONLY fields from profile_dataset (no computations).

    If profiling was skipped or validation failed:
      - Produce a short report using this template:
        # Dataset Quality Report
        ## Validation Failed
        The dataset could not be profiled.
        ### Issues
        - {each validation_error}
        ### Warnings
        - {each warning}
      - Include note: "Profiling skipped due to validation failure."

    Otherwise, output sections in EXACT order:
      1) # Dataset Quality Report
      2) ## Overview
      3) ## Schema Summary
      4) ## Missingness Analysis
      5) ## Duplicate Analysis
      6) ## Summary Statistics (table)
      7) ## Data Quality Score
      8) ## Recommendations
      9) ## Validation Issues (only if warnings or validation_errors exist)

    Recommendations (rule-based, no math):
      - If any column missingness > 30%: suggest median imputation or removing the column.
      - If duplicate ratio > 10% (use values already present in profile_dataset): suggest investigating duplicate generation rules.
      - If outliers are flagged in profile_dataset outputs: suggest validating business logic for extreme values.

    Guardrail:
      - Reporter must NOT compute numeric values. If computation is attempted, add warning
        "reporter_attempted_computation".
    ''',
  "expected_output": '''
    Markdown report following the above schema.
    ''',
  "agent": "reporter",
  # "context": [profile_dataset],
  "markdown": True,
  "output_file": "output/{outdata_dir}/report.md"
}

In [15]:
load_dataset_task=Task(
    description=load_dataset_config.get("description"),
    expected_output=load_dataset_config.get("expected_output"),
    agent=load_dataset_config.get("agent"),
    output_file=load_dataset_config.get("output_file"),
    output_pydantic=DatasetLoadOut
)

profile_dataset_task=Task(
    description=profile_dataset_config.get("description"),
    expected_output=profile_dataset_config.get("expected_output"),
    agent=profile_dataset_config.get("agent"),
    context=load_dataset_task,
    markdown=profile_dataset_config.get("markdown"),
    output_file=profile_dataset_config.get("output_file"),
    output_pydantic=DatasetProfile
)

write_report_task=Task(
    description=write_report_config.get("description"),
    expected_output=write_report_config.get("expected_output"),
    agent=write_report_config.get("agent"),
    context=profile_dataset_task,
    markdown=write_report_config.get("markdown"),
    output_file=write_report_config.get("output_file"),
)



AttributeError: 'str' object has no attribute 'get'

In [ ]:
@CrewBase
class DatasetProfiler():
    """DatasetProfiler crew"""
    # -------------------------
    # Agents
    # -------------------------

    @agent
    def loader(self) -> Agent: return loader_agent

    @agent
    def profiler(self) -> Agent: return profiler_agent

    @agent
    def reporter(self) -> Agent: return reporter_agent
    
    # -------------------------
    # Tasks
    # -------------------------

    @task
    def load_dataset(self) -> Task: return load_dataset_task

    @task
    def profile_dataset(self) -> Task: return profile_dataset_task

    @task
    def write_report(self) -> Task: return write_report_task

    # -------------------------
    # Crew
    # -------------------------

    @crew
    def crew(self) -> Crew:
        """Creates the DatasetProfiler crew"""

        return Crew(
            name="DatasetQualityProfiler",
            agents=self.agents,
            tasks=self.tasks,
            process=Process.sequential,
            verbose=True,
            tracing=True
        )


## Running the crew

![1](../img/img_DataQualityProfiler/crew-1.png)
![2](../img/img_DataQualityProfiler/crew-2.png)
![3](../img/img_DataQualityProfiler/crew-3.png)
![4](../img/img_DataQualityProfiler/crew-4.png)
![5](../img/img_DataQualityProfiler/crew-5.png)
![6](../img/img_DataQualityProfiler/crew-6.png)
...
![7](../img/img_DataQualityProfiler/crew-7.png)
...
![8](../img/img_DataQualityProfiler/crew-8.png)



See the output generated: [report]()